In [25]:
## Random forest Regressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [26]:
df = pd.read_csv('cardekho_imputated.csv')
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [27]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [28]:
df.drop(columns=['car_name','brand','Unnamed: 0'], axis=1, inplace=True)
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [29]:
len(df['model'].unique())

120

In [30]:
## Getting all different types of feature

numerical_features = [features for features in df.columns if df[features].dtype != 'O']
print('Number of Numerical Features' , len(numerical_features))
categorical_features = [features for features in df.columns if df[features].dtype == 'O']
print('Number of categorical features', len(categorical_features))
discrete_features = [features for features in numerical_features if len(df[features].unique()) <= 25]
print('Number of Discrete features', len(discrete_features))
categorical_features = [features for features in numerical_features if features not in discrete_features]
print('NUmber of categorical', len(categorical_features))

Number of Numerical Features 7
Number of categorical features 4
Number of Discrete features 2
NUmber of categorical 5


In [31]:
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [32]:
X.shape

(15411, 10)

In [33]:
## feature encoding and scalling
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [34]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [37]:
len(df['fuel_type'].unique()), len(df['seller_type'].unique()), len(df['transmission_type'].unique())

(5, 3, 2)

In [40]:
numerical_features = X.select_dtypes(exclude='object').columns
oh_columns = ['fuel_type', 'seller_type', 'transmission_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer, oh_columns),
        ('StandardScaler', num_transformer, numerical_features)
    ]
)


In [41]:
preprocessor

,transformers,"[('OneHotEncoder', ...), ('StandardScaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [42]:
X = preprocessor.fit_transform(X)


In [43]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [45]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.33,random_state=42)
X_train.shape

(10325, 14)

In [48]:
## Model Training
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error 

In [49]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [58]:
models = {
    'Linear Regression' : LinearRegression(),
    'K Nearest' : KNeighborsRegressor(),
    'Ridge' : Ridge(),
    'Lasso' : Lasso(),
    'Random Forest' : RandomForestRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make Model prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Evaluate Model
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    ## print train accuracy
    print(list(models.keys())[i])
    
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 563572.9972
- Mean Absolute Error: 269089.2604
- R2 Score: 0.6170
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 503390.2467
- Mean Absolute Error: 281003.4130
- R2 Score: 0.6569


K Nearest
Model performance for Training set
- Root Mean Squared Error: 344663.7933
- Mean Absolute Error: 92993.2397
- R2 Score: 0.8567
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 279025.7422
- Mean Absolute Error: 115558.5381
- R2 Score: 0.8946


Ridge
Model performance for Training set
- Root Mean Squared Error: 563573.8496
- Mean Absolute Error: 269042.5654
- R2 Score: 0.6170
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 503371.9238
- Mean Absolute Error: 280947.9075
- R2 Score: 0.6570


Lasso
Model performance for Training set
- Root Mean Squared Error: 563573.0071
- Mean Absolute Erro

In [53]:
## Hyperparameter Tunning
knn_params = {'n_neighbors' : [2, 5, 10, 20, 40]}
rf_params = {
            "max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]
}

In [54]:
randomcv_models = [
   ('KNN', KNeighborsRegressor(), knn_params),
    ("RF", RandomForestRegressor(), rf_params)
]

In [55]:
from sklearn.model_selection import RandomizedSearchCV
model_param = {}

for name, model, params in randomcv_models:
    random = RandomizedSearchCV(
                                   estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1
    )
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

    for model_name in model_param:
        print(f'----------- Best Model {model_name} -----------')
        print(model_param[model_name])


Fitting 3 folds for each of 5 candidates, totalling 15 fits
----------- Best Model KNN -----------
{'n_neighbors': 5}
Fitting 3 folds for each of 100 candidates, totalling 300 fits
----------- Best Model KNN -----------
{'n_neighbors': 5}
----------- Best Model RF -----------
{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 7, 'max_depth': None}


In [ ]:
models = {
    
    'K Nearest' : KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'Random Forest' : RandomForestRegressor(n_estimators = 200, min_samples_split = 2, max_features = 7, max_depth = None, n_jobs=-1)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make Model prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Model Test Performance
    model_train_accuracy = accuracy_score(y_test, y_test_pred)
    model_train_f1 = f1_score(y_test,y_test_pred,average='weighted')
    model_train_precision = precision_score(y_test, y_test_pred)
    model_train_recall = recall_score(y_test, y_test_pred)
    model_train_rocauc_score = roc_auc_score(y_test, y_test_pred)

    ## Evaluate Model
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    ## print train accuracy
    print(list(models.keys())[i])
    
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

K Nearest
Model performance for Training set
- Root Mean Squared Error: 344663.7933
- Mean Absolute Error: 92993.2397
- R2 Score: 0.8567
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 344663.7933
- Mean Absolute Error: 92993.2397
- R2 Score: 0.8567


Random Forest
Model performance for Training set
- Root Mean Squared Error: 134303.9640
- Mean Absolute Error: 39594.1187
- R2 Score: 0.9782
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 134303.9640
- Mean Absolute Error: 39594.1187
- R2 Score: 0.9782


